In [13]:
# -*- coding: utf-8 -*-
"""
CO modeling with Fourier-Residual LSTM + Matplotlib exports
- Null fixing, daily aggregation
- 80:20 chronological split
- R2, MAE, MSE, RMSE
- 7-day forecast
- Saves 5 figures:
  1) histogram.png
  2) time_series_baseline.png
  3) forecast_7d.png
  4) learning_curve.png
  5) actual_vs_pred.png
"""

import os
import numpy as np
import pandas as pd
from datetime import timedelta
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# -----------------------------
# Paths & output directory
# -----------------------------
CSV_PATH = "CO_hex30_urban.csv"   # change if needed
OUT_DIR  = "./co_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# 1) Load & daily aggregation
# -----------------------------
df = pd.read_csv(CSV_PATH)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
df['CO'] = pd.to_numeric(df['CO'], errors='coerce')

# Daily mean across all hex cells
daily = (df.groupby('date', as_index=True)['CO']
         .mean()
         .sort_index()
         .to_frame())

# --------------------------------------
# 2) Fix nulls & regularize daily index
# --------------------------------------
full_idx = pd.date_range(daily.index.min(), daily.index.max(), freq='D')
daily = daily.reindex(full_idx)
daily.index.name = 'date'

# Interpolate internal gaps in time, then fill edges; final guard = median
daily['CO'] = daily['CO'].interpolate(method='time', limit_direction='both')
if daily['CO'].isna().any():
    daily['CO'] = daily['CO'].fillna(daily['CO'].median())

# ---------------------------
# Descriptive statistics
# ---------------------------
desc = daily['CO'].describe().to_frame('CO_statistics')
desc.loc['missing_count_in_raw'] = df['CO'].isna().sum()
desc.loc['start_date'] = str(daily.index.min().date())
desc.loc['end_date']   = str(daily.index.max().date())
desc.to_csv(os.path.join(OUT_DIR, "co_descriptive_stats.csv"))

# ---------------------------------
# 3) Train/Test split (chronology)
# ---------------------------------
n = len(daily)
split_idx = int(n * 0.8)
train_df = daily.iloc[:split_idx].copy()
test_df  = daily.iloc[split_idx:].copy()

# ---------------------------------------------------------
# 4) Fourier baseline (my “formula” seasonal/trend model)
# ---------------------------------------------------------
def make_fourier_features(dates: pd.DatetimeIndex):
    t = (dates - dates[0]).days.values.astype(float)
    periods = [7.0, 365.25]  # weekly + annual
    X = np.ones((len(t), 1))  # intercept
    for p in periods:
        X = np.column_stack([
            X,
            np.sin(2*np.pi * t / p),
            np.cos(2*np.pi * t / p)
        ])
    return X, periods

X_train, periods = make_fourier_features(train_df.index)
y_train = train_df['CO'].values.reshape(-1, 1)

# Fit by least squares on training only (prevents leakage)
beta, *_ = np.linalg.lstsq(X_train, y_train, rcond=None)

def baseline_from_dates(dates):
    X, _ = make_fourier_features(dates)
    return (X @ beta).reshape(-1)

daily['baseline'] = baseline_from_dates(daily.index)
daily['residual'] = daily['CO'] - daily['baseline']

train_df['baseline'] = daily['baseline'].iloc[:split_idx].values
test_df['baseline']  = daily['baseline'].iloc[split_idx:].values
train_df['residual'] = daily['residual'].iloc[:split_idx].values
test_df['residual']  = daily['residual'].iloc[split_idx:].values

# ------------------------------------------------
# 5) Build supervised sequences for LSTM on r_t
# ------------------------------------------------
def to_sequences(series: np.ndarray, lookback: int):
    Xs, ys = [], []
    for i in range(len(series) - lookback):
        Xs.append(series[i:i+lookback])
        ys.append(series[i+lookback])
    return np.array(Xs), np.array(ys)

lookback = 21  # ~3 weeks
scaler = MinMaxScaler()
train_resid_scaled = scaler.fit_transform(train_df[['residual']].values)
test_resid_scaled  = scaler.transform(test_df[['residual']].values)

X_train_seq, y_train_seq = to_sequences(train_resid_scaled.flatten(), lookback)
X_test_seq,  y_test_seq  = to_sequences(test_resid_scaled.flatten(), lookback)

X_train_seq = X_train_seq.reshape(-1, lookback, 1)
X_test_seq  = X_test_seq.reshape(-1, lookback, 1)

# ------------------------------
# 6) Define & train LSTM
# ------------------------------
tf.keras.utils.set_random_seed(42)

model = Sequential([
    LSTM(64, input_shape=(lookback, 1), return_sequences=True),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.1),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='mse',
    metrics=[tf.keras.metrics.MeanAbsoluteError(name='mae')]
)

callbacks = [
    EarlyStopping(patience=20, restore_best_weights=True),
    ReduceLROnPlateau(patience=8, factor=0.5)
]

history = model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_test_seq, y_test_seq),
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# --------------------------------------------
# 7) Evaluation on test (R2, MAE, MSE, RMSE)
# --------------------------------------------
resid_pred_test_scaled = model.predict(X_test_seq, verbose=0).flatten()
resid_pred_test = scaler.inverse_transform(resid_pred_test_scaled.reshape(-1,1)).flatten()

# Align with test dates after lookback
test_dates_aligned = test_df.index[lookback:]
baseline_aligned   = test_df['baseline'].values[lookback:]
y_true_aligned     = test_df['CO'].values[lookback:]
y_pred_aligned     = baseline_aligned + resid_pred_test

r2  = r2_score(y_true_aligned, y_pred_aligned)
mae = mean_absolute_error(y_true_aligned, y_pred_aligned)
mse = mean_squared_error(y_true_aligned, y_pred_aligned)
rmse = np.sqrt(mse)

metrics = pd.DataFrame({
    'metric': ['R2', 'MAE', 'MSE', 'RMSE'],
    'value': [r2, mae, mse, rmse]
})
metrics.to_csv(os.path.join(OUT_DIR, "co_metrics.csv"), index=False)
print(metrics)

# -----------------------------------
# 8) 7-day forecast (recursive)
# -----------------------------------
last_date = daily.index.max()
future_dates = pd.date_range(last_date + timedelta(days=1), periods=7, freq='D')

# prepare baseline for future
future_baseline = baseline_from_dates(future_dates)

# recursive residual forecast
full_resid_scaled = scaler.transform(daily[['residual']].values).flatten()
window = full_resid_scaled[-lookback:].tolist()

future_resid_scaled = []
for _ in range(7):
    x = np.array(window[-lookback:]).reshape(1, lookback, 1)
    yhat = model.predict(x, verbose=0).item()
    future_resid_scaled.append(yhat)
    window.append(yhat)

future_resid = scaler.inverse_transform(np.array(future_resid_scaled).reshape(-1,1)).flatten()
future_pred  = future_baseline + future_resid

forecast_df = pd.DataFrame({
    'date': future_dates,
    'baseline': future_baseline,
    'residual_pred': future_resid,
    'CO_pred': future_pred
})
forecast_df.to_csv(os.path.join(OUT_DIR, "co_forecast_7d.csv"), index=False)

# =====================================================
# 9) CHARTS (Matplotlib; each chart is a separate plot)
#    NOTE: Do not set explicit colors or styles per spec
# =====================================================

# (1) Histogram – distribution of daily mean CO
plt.figure(figsize=(15, 7))
daily['CO'].hist(bins=30)
plt.title("Distribution of Daily Mean CO")
plt.xlabel("CO")
plt.ylabel("Frequency")
plt.tight_layout()
# plt.savefig(os.path.join(OUT_DIR, "histogram.png"), dpi=200)
# plt.close()
plt.show()

# (2) Time series – actual daily CO vs Fourier baseline
plt.figure(figsize=(15, 7))
plt.plot(daily.index, daily['CO'], label="CO (daily mean)")
plt.plot(daily.index, daily['baseline'], label="Fourier baseline (fit)", linestyle="--")
plt.title("Daily CO vs Baseline Fit")
plt.xlabel("Date"); plt.ylabel("CO")
plt.legend()
plt.tight_layout()
# plt.savefig(os.path.join(OUT_DIR, "time_series_baseline.png"), dpi=200)
# plt.close()
plt.show()

# (3) 7-day forecast – historical, baseline, projected values
plt.figure(figsize=(15, 7))
plt.plot(daily.index, daily['CO'], label="Historical CO")
plt.plot(daily.index, daily['baseline'], label="Baseline (fit)", linestyle="--")
plt.plot(future_dates, future_pred, marker="o", label="7-day Forecast")
plt.title("7-day CO Forecast (Fourier-Residual LSTM)")
plt.xlabel("Date"); plt.ylabel("CO")
plt.legend()
plt.tight_layout()
# plt.savefig(os.path.join(OUT_DIR, "forecast_7d.png"), dpi=200)
# plt.close()
plt.show()

# (4) Learning curve
plt.figure(figsize=(15, 7))
plt.plot(history.history['loss'], label="train_loss")
plt.plot(history.history['val_loss'], label="val_loss")
plt.title("Learning Curve (Loss)")
plt.xlabel("Epoch"); plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
# plt.savefig(os.path.join(OUT_DIR, "learning_curve.png"), dpi=200)
# plt.close()
plt.show()

# (5) Actual vs predicted on test
plt.figure(figsize=(15, 7))
plt.plot(test_dates_aligned, y_true_aligned, label="Actual CO")
plt.plot(test_dates_aligned, y_pred_aligned, label="Predicted CO")
plt.title("Actual vs Predicted CO (Test)")
plt.xlabel("Date"); plt.ylabel("CO")
plt.legend()
plt.tight_layout()
# plt.savefig(os.path.join(OUT_DIR, "actual_vs_pred.png"), dpi=200)
# plt.close()
plt.show()

print("All outputs saved to:", os.path.abspath(OUT_DIR))


Epoch 1/200


/Users/sakdahomhuan/miniforge3/envs/udfire/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 92ms/step - loss: 0.1075 - mae: 0.2863 - val_loss: 0.0432 - val_mae: 0.1821 - learning_rate: 0.0010
Epoch 2/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0590 - mae: 0.1939 - val_loss: 0.1042 - val_mae: 0.2860 - learning_rate: 0.0010
Epoch 3/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0312 - mae: 0.1257 - val_loss: 0.2344 - val_mae: 0.4439 - learning_rate: 0.0010
Epoch 4/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0304 - mae: 0.1364 - val_loss: 0.1547 - val_mae: 0.3546 - learning_rate: 0.0010
Epoch 5/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0232 - mae: 0.1117 - val_loss: 0.1001 - val_mae: 0.2798 - learning_rate: 0.0010
Epoch 6/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0274 - mae: 0.1218 - val_loss: 0.1054 - val_mae: 0.2870 - learning_rate: 0.0010
Epoch 7/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0256 - mae: 0.1147 - val_loss: 0.1368 - val_mae: 0.3322 - learning_rate: 0.0010
Epoch 8/200
4/4 ━━━━━━━━━━━━━━━━

/var/folders/qm/p5vffjb56gvb80sz8bwjxy780000gn/T/ipykernel_13786/1333173305.py:237: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/qm/p5vffjb56gvb80sz8bwjxy780000gn/T/ipykernel_13786/1333173305.py:249: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/qm/p5vffjb56gvb80sz8bwjxy780000gn/T/ipykernel_13786/1333173305.py:262: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/qm/p5vffjb56gvb80sz8bwjxy780000gn/T/ipykernel_13786/1333173305.py:274: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/qm/p5vffjb56gvb80sz8bwjxy780000gn/T/ipykernel_13786/1333173305.py:286: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
